# 05_evaluation

In [ ]:
# ============================================================
# FINAL AFTER-EDIT CHECK — ALL EVALUATION SAMPLES
#
# IMPORTANT:
#   Run IMMEDIATELY after Cell 9 while model is at WT.
#   DO NOT run restore_all_W0() before this cell.
#
# FIX:
#   valid_df is NOT used here.
#
#   Reason:
#   valid_df intentionally excludes invalid W0 model outputs,
#   while final evaluation must include ALL evaluation samples.
#
#   Any sample not already prepared in groups / GLOBAL_PROTECT
#   is reconstructed directly with:
#
#       build_example(sample_id, true_label)
#
# ============================================================

import pandas as pd
import numpy as np


print("=" * 80)
print("FINAL AFTER-EDIT GENERATED EVALUATION")
print("=" * 80)


# ============================================================
# 1. Load original W0 predictions
# ============================================================

before_df = pd.read_csv(
    run_path(
        "HUMAID_all_predictions_per_instance.csv"
    )
).copy()


# ------------------------------------------------------------
# Exact experimental scope
# ------------------------------------------------------------

if "shot" in before_df.columns:

    before_df = before_df[
        before_df["shot"]
        .astype(str)
        .str.lower()
        == "one"
    ].copy()


if "prompt" in before_df.columns:

    before_df = before_df[
        before_df["prompt"]
        .astype(str)
        .str.lower()
        == "domain"
    ].copy()


if "iteration" in before_df.columns:

    before_df = before_df[
        before_df["iteration"]
        .astype(int)
        == 0
    ].copy()


# ------------------------------------------------------------
# Normalize IDs
# ------------------------------------------------------------

before_df["sample_id"] = (
    before_df["sample_id"]
    .astype(str)
)


# ------------------------------------------------------------
# One row per evaluation sample
# ------------------------------------------------------------

before_df = (
    before_df
    .drop_duplicates(
        subset=["sample_id"],
        keep="first"
    )
    .reset_index(drop=True)
)


print(
    "Evaluation samples:",
    len(before_df)
)


assert len(before_df) > 0, (
    "No evaluation samples found."
)


# ============================================================
# 2. Collect already prepared examples
# ============================================================

prepared_by_id = {}


# ------------------------------------------------------------
# 2A. Ripple members + per-group protect
# ------------------------------------------------------------

for g in groups:

    for ex in g.get(
        "members",
        []
    ):

        sid = str(
            ex["sample_id"]
        )

        prepared_by_id[sid] = ex


    for ex in g.get(
        "protect",
        []
    ):

        sid = str(
            ex["sample_id"]
        )

        if sid not in prepared_by_id:

            prepared_by_id[sid] = ex


# ------------------------------------------------------------
# 2B. Also reuse GLOBAL_PROTECT
# ------------------------------------------------------------

if "GLOBAL_PROTECT" in globals():

    for ex in GLOBAL_PROTECT:

        sid = str(
            ex["sample_id"]
        )

        if sid not in prepared_by_id:

            prepared_by_id[sid] = ex


print(
    "Already prepared examples:",
    len(prepared_by_id)
)


# ============================================================
# 3. Prepare EVERY missing evaluation sample
#
# IMPORTANT:
#   DO NOT use valid_df.
# ============================================================

required_columns = {
    "sample_id",
    "true_label",
}


missing_columns = (
    required_columns
    -
    set(before_df.columns)
)


assert len(missing_columns) == 0, (
    f"before_df missing columns: "
    f"{sorted(missing_columns)}"
)


eval_lookup = {

    str(r["sample_id"]):
        r

    for _, r in before_df.iterrows()

}


newly_built = 0


for sid in before_df["sample_id"]:

    sid = str(sid)


    if sid in prepared_by_id:

        continue


    assert sid in eval_lookup, (
        f"sample_id={sid} missing from before_df lookup"
    )


    row = eval_lookup[sid]


    true_label = str(
        row["true_label"]
    )


    ex = build_example(
        sid,
        true_label
    )


    assert ex is not None, (
        f"build_example returned None "
        f"for sample_id={sid}"
    )


    assert (
        str(ex["sample_id"])
        ==
        sid
    ), (
        f"sample_id mismatch: "
        f"expected={sid}, "
        f"got={ex['sample_id']}"
    )


    prepared_by_id[sid] = ex

    newly_built += 1


print(
    "Newly built examples:",
    newly_built
)


print(
    "Total prepared examples:",
    len(prepared_by_id)
)


# ============================================================
# 4. Ensure ALL evaluation samples are available
# ============================================================

evaluation_ids = set(
    before_df["sample_id"]
    .astype(str)
)


prepared_ids = set(
    prepared_by_id.keys()
)


missing = (
    evaluation_ids
    -
    prepared_ids
)


assert len(missing) == 0, (
    f"Missing prepared examples: "
    f"{sorted(missing)}"
)


print(
    "All evaluation samples prepared:",
    len(evaluation_ids)
)


# ============================================================
# 5. Generate AFTER predictions at WT
#
# IMPORTANT:
#   model must still be physically at WT here.
# ============================================================

after_rows = []


for i, row in before_df.iterrows():

    sid = str(
        row["sample_id"]
    )


    ex = prepared_by_id[
        sid
    ]


    pred = generated_prediction(
        ex
    )


    after_rows.append(
        {
            "sample_id":
                sid,

            "predicted_label_WT":
                pred[
                    "predicted_label"
                ],

            "raw_output_WT":
                pred[
                    "raw_output"
                ],

            "correct_WT":
                bool(
                    pred[
                        "correct"
                    ]
                ),
        }
    )


    if (
        (i + 1) % 10 == 0
        or
        i + 1 == len(before_df)
    ):

        print(
            f"Generated: "
            f"{i + 1}/"
            f"{len(before_df)}"
        )


after_df = pd.DataFrame(
    after_rows
)


# ------------------------------------------------------------
# Sanity
# ------------------------------------------------------------

assert len(after_df) == len(before_df)


assert (
    after_df["sample_id"]
    .nunique()
    ==
    len(before_df)
), (
    "Duplicate or missing sample IDs "
    "in WT predictions."
)


# ============================================================
# 6. Merge W0 and WT
# ============================================================

df = before_df.merge(
    after_df,
    on="sample_id",
    how="left",
    validate="one_to_one"
)


assert len(df) == len(before_df)


assert (
    df["predicted_label_WT"]
    .notna()
    .all()
), (
    "Some WT predictions are missing."
)


# ============================================================
# 7. Determine W0 correctness
# ============================================================

if "is_correct" in df.columns:

    df["correct_W0"] = (
        df["is_correct"]
        .astype(str)
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )


elif "is_misclassified" in df.columns:

    df["correct_W0"] = ~(
        df["is_misclassified"]
        .astype(str)
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )


else:

    # Fallback: derive directly from prediction vs true label
    if (
        "predicted_label" in df.columns
        and
        "true_label" in df.columns
    ):

        df["correct_W0"] = (
            df["predicted_label"]
            .astype(str)
            .str.strip()
            .str.lower()
            ==
            df["true_label"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    else:

        raise KeyError(
            "Cannot determine W0 correctness. "
            "Need is_correct, is_misclassified, "
            "or predicted_label + true_label."
        )


df["correct_WT"] = (
    df["correct_WT"]
    .astype(bool)
)


# ============================================================
# 8. W0 -> WT transition
# ============================================================

def transition(row):

    before = bool(
        row["correct_W0"]
    )

    after = bool(
        row["correct_WT"]
    )


    if (
        not before
        and
        after
    ):

        return "FIXED"


    if (
        not before
        and
        not after
    ):

        return "STILL_WRONG"


    if (
        before
        and
        after
    ):

        return "PRESERVED"


    return "DAMAGED"


df["transition"] = df.apply(
    transition,
    axis=1
)


# ============================================================
# 9. Counts
# ============================================================

counts = (
    df["transition"]
    .value_counts()
)


FIXED = int(
    counts.get(
        "FIXED",
        0
    )
)


STILL_WRONG = int(
    counts.get(
        "STILL_WRONG",
        0
    )
)


PRESERVED = int(
    counts.get(
        "PRESERVED",
        0
    )
)


DAMAGED = int(
    counts.get(
        "DAMAGED",
        0
    )
)


N = len(df)


correct_W0 = int(
    df["correct_W0"]
    .sum()
)


wrong_W0 = (
    N
    -
    correct_W0
)


correct_WT = int(
    df["correct_WT"]
    .sum()
)


wrong_WT = (
    N
    -
    correct_WT
)


acc_W0 = (
    correct_W0
    /
    N
)


acc_WT = (
    correct_WT
    /
    N
)


# ============================================================
# 10. Accounting sanity
# ============================================================

assert (
    correct_W0
    ==
    PRESERVED
    +
    DAMAGED
)


assert (
    wrong_W0
    ==
    FIXED
    +
    STILL_WRONG
)


assert (
    correct_WT
    ==
    PRESERVED
    +
    FIXED
)


assert (
    wrong_WT
    ==
    STILL_WRONG
    +
    DAMAGED
)


assert (
    N
    ==
    FIXED
    +
    STILL_WRONG
    +
    PRESERVED
    +
    DAMAGED
)


print(
    "Accounting checks passed."
)


# ============================================================
# 11. Final report
# ============================================================

print()

print("=" * 80)

print(
    "FINAL W0 -> WT RESULTS"
)

print("=" * 80)


print(
    f"Total               : {N}"
)


print()

print(
    f"Correct W0          : {correct_W0}"
)

print(
    f"Wrong W0            : {wrong_W0}"
)


print()

print(
    f"Correct WT          : {correct_WT}"
)

print(
    f"Wrong WT            : {wrong_WT}"
)


print()

print(
    f"FIXED               : {FIXED}"
)

print(
    f"STILL_WRONG         : {STILL_WRONG}"
)

print(
    f"PRESERVED           : {PRESERVED}"
)

print(
    f"DAMAGED             : {DAMAGED}"
)


print()

print(
    f"Accuracy W0         : "
    f"{acc_W0:.6f}"
)

print(
    f"Accuracy WT         : "
    f"{acc_WT:.6f}"
)

print(
    f"Accuracy change     : "
    f"{acc_WT - acc_W0:+.6f}"
)


print()


if wrong_W0 > 0:

    edit_success = (
        FIXED
        /
        wrong_W0
    )

    print(
        f"Edit Success        : "
        f"{edit_success:.4f}"
    )

else:

    edit_success = np.nan

    print(
        "Edit Success        : nan"
    )


if correct_W0 > 0:

    preservation = (
        PRESERVED
        /
        correct_W0
    )

    damage_rate = (
        DAMAGED
        /
        correct_W0
    )


    print(
        f"Preservation        : "
        f"{preservation:.4f}"
    )


    print(
        f"Damage Rate         : "
        f"{damage_rate:.4f}"
    )

else:

    preservation = np.nan
    damage_rate = np.nan


    print(
        "Preservation        : nan"
    )


    print(
        "Damage Rate         : nan"
    )


# ============================================================
# 12. Save detailed table
# ============================================================

FINAL_PATH = run_path(
    "HumAID_FINAL_W0_WT_generated.csv"
)


df.to_csv(
    FINAL_PATH,
    index=False
)


print()

print(
    "Saved detailed W0-WT table:"
)

print(
    FINAL_PATH
)


# ============================================================
# 13. Save compact metrics table
# ============================================================

metrics_df = pd.DataFrame(
    [
        {
            "total":
                N,

            "correct_W0":
                correct_W0,

            "wrong_W0":
                wrong_W0,

            "correct_WT":
                correct_WT,

            "wrong_WT":
                wrong_WT,

            "fixed":
                FIXED,

            "still_wrong":
                STILL_WRONG,

            "preserved":
                PRESERVED,

            "damaged":
                DAMAGED,

            "accuracy_W0":
                acc_W0,

            "accuracy_WT":
                acc_WT,

            "accuracy_change":
                acc_WT - acc_W0,

            "edit_success":
                edit_success,

            "preservation":
                preservation,

            "damage_rate":
                damage_rate,
        }
    ]
)


METRICS_PATH = run_path(
    "HumAID_FINAL_W0_WT_metrics.csv"
)


metrics_df.to_csv(
    METRICS_PATH,
    index=False
)


print()

print(
    "Saved compact metrics:"
)

print(
    METRICS_PATH
)


print()

print(
    "FINAL AFTER-EDIT evaluation complete."
)

print(
    "ALL evaluation samples included."
)

print(
    "IMPORTANT: model is still at WT."
)

FINAL AFTER-EDIT GENERATED EVALUATION
Evaluation samples: 500
Already prepared examples: 498
Newly built examples: 2
Total prepared examples: 500
All evaluation samples prepared: 500
Generated: 10/500
Generated: 20/500
Generated: 30/500
Generated: 40/500
Generated: 50/500
Generated: 60/500
Generated: 70/500
Generated: 80/500
Generated: 90/500
Generated: 100/500
Generated: 110/500
Generated: 120/500
Generated: 130/500
Generated: 140/500
Generated: 150/500
Generated: 160/500
Generated: 170/500
Generated: 180/500
Generated: 190/500
Generated: 200/500
Generated: 210/500
Generated: 220/500
Generated: 230/500
Generated: 240/500
Generated: 250/500
Generated: 260/500
Generated: 270/500
Generated: 280/500
Generated: 290/500
Generated: 300/500
Generated: 310/500
Generated: 320/500
Generated: 330/500
Generated: 340/500
Generated: 350/500
Generated: 360/500
Generated: 370/500
Generated: 380/500
Generated: 390/500
Generated: 400/500
Generated: 410/500
Generated: 420/500
Generated: 430/500
Generated

In [ ]:
# ============================================================
# FINAL HARVEST — CURRENT WT MODEL
# Retained metrics only:
# N, Accuracy, Precision (macro), Recall (macro), Macro-F1,
# MCC, and Macro OvR ROC-AUC — for W0 and WT.
#
# Original metric formulas and ROC-AUC implementation unchanged.
# Run while the edited model is still at WT.
# ============================================================

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
import torch
from transformers import AutoModelForCausalLM

# ============================================================
# 0. PATHS
# ============================================================

# Use the run directory selected in Global Configuration.
# Never override it with an old hard-coded CSC path.
assert "RUN_DIR" in globals(), (
    "RUN_DIR is not defined. Run the Global Configuration cell first."
)
RUN_DIR = Path(RUN_DIR).resolve()
assert RUN_DIR.is_dir(), f"Current RUN_DIR does not exist: {RUN_DIR}"

FINAL_DIR = RUN_DIR / "FINAL_ALL_METRICS"
FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 100)
print("FINAL HARVEST — CURRENT WT")
print("=" * 100)

print("RUN_DIR  :", RUN_DIR)
print("SAVE_DIR :", FINAL_DIR)


# ============================================================
# 1. SAVE CURRENT WT MODEL BEFORE ANY RESET
# ============================================================

print("\n" + "=" * 100)
print("1. SAVING CURRENT WT MODEL")
print("=" * 100)

MODEL_OBJ = None
MODEL_NAME_IN_MEMORY = None

for candidate in [
    "model",
    "model_edited",
    "edited_model",
]:

    if (
        candidate in globals()
        and globals()[candidate] is not None
    ):
        MODEL_OBJ = globals()[candidate]
        MODEL_NAME_IN_MEMORY = candidate
        break


if MODEL_OBJ is not None:

    WT_MODEL_DIR = (
        RUN_DIR
        / "FINAL_WT_MODEL"
    )

    WT_MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    try:

        MODEL_OBJ.save_pretrained(
            WT_MODEL_DIR,
            safe_serialization=True
        )

        print(
            f"[OK] WT model saved from variable "
            f"`{MODEL_NAME_IN_MEMORY}`"
        )

        print(WT_MODEL_DIR)

        # tokenizer
        if (
            "tokenizer" in globals()
            and tokenizer is not None
        ):

            tokenizer.save_pretrained(
                WT_MODEL_DIR
            )

            print("[OK] tokenizer saved")

    except Exception as e:

        print(
            "[WARN] save_pretrained failed:"
        )

        print(e)

        # fallback
        try:

            import torch

            fallback_path = (
                WT_MODEL_DIR
                / "WT_state_dict.pt"
            )

            torch.save(
                MODEL_OBJ.state_dict(),
                fallback_path
            )

            print(
                "[OK] WT state_dict saved instead:"
            )

            print(fallback_path)

        except Exception as e2:

            print(
                "[ERROR] Could not save WT model:"
            )

            print(e2)

else:

    print(
        "[WARN] No model/model_edited variable found."
    )


# ============================================================
# 2. GET FINAL W0 -> WT DATAFRAME
# ============================================================

print("\n" + "=" * 100)
print("2. LOADING FINAL W0 -> WT RESULTS")
print("=" * 100)

FINAL_PRED_PATH = (
    RUN_DIR
    / "HumAID_FINAL_W0_WT_generated.csv"
)


FINAL_PRED_PATH = (
    RUN_DIR / "HumAID_FINAL_W0_WT_generated.csv"
)

assert FINAL_PRED_PATH.is_file(), (
    f"Current run predictions not found: {FINAL_PRED_PATH}"
)

final_df = pd.read_csv(FINAL_PRED_PATH)

required = {
    "sample_id",
    "true_label",
    "predicted_label_WT"
}

assert required.issubset(final_df.columns)
assert final_df["sample_id"].is_unique
assert final_df["predicted_label_WT"].notna().all()

print("[OK] Loaded current run predictions:", FINAL_PRED_PATH)
print("Evaluation rows:", len(final_df))
# ============================================================
# 3. NORMALIZE LABELS
# ============================================================

def norm_label(x):

    if pd.isna(x):
        return "__wrong__"

    x = str(x).strip().lower()

    x = re.sub(
        r"[\s\-]+",
        "_",
        x
    )

    x = re.sub(
        r"[^a-z0-9_]",
        "",
        x
    )

    x = re.sub(
        r"_+",
        "_",
        x
    ).strip("_")

    return (
        x
        if x
        else "__wrong__"
    )


assert "true_label" in final_df.columns
assert "predicted_label_WT" in final_df.columns


if "predicted_label_W0" in final_df.columns:

    W0_COL = "predicted_label_W0"

elif "predicted_label" in final_df.columns:

    W0_COL = "predicted_label"

else:

    raise KeyError(
        "W0 prediction column not found."
    )


WT_COL = "predicted_label_WT"


y_true = (
    final_df["true_label"]
    .map(norm_label)
)

y_w0 = (
    final_df[W0_COL]
    .map(norm_label)
)

y_wt = (
    final_df[WT_COL]
    .map(norm_label)
)


# ============================================================
# 4. OVERALL TASK PERFORMANCE
# ============================================================

print("\n" + "=" * 100)
print("3. OVERALL TASK PERFORMANCE")
print("=" * 100)


def cls_metrics(
    yt,
    yp
):

    return {

        "Accuracy":
            accuracy_score(
                yt,
                yp
            ),

        "Precision_Macro":
            precision_score(
                yt,
                yp,
                average="macro",
                zero_division=0
            ),

        "Recall_Macro":
            recall_score(
                yt,
                yp,
                average="macro",
                zero_division=0
            ),

        "Macro_F1":
            f1_score(
                yt,
                yp,
                average="macro",
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                yt,
                yp
            ),
    }


score_w0 = cls_metrics(
    y_true,
    y_w0
)

score_wt = cls_metrics(
    y_true,
    y_wt
)


overall_rows = []

for metric in score_w0:

    overall_rows.append(
        {
            "Metric":
                metric,

            "Before_W0":
                score_w0[metric],

            "After_WT":
                score_wt[metric],

            "Delta":
                score_wt[metric]
                -
                score_w0[metric],
        }
    )


overall_df = pd.DataFrame(
    overall_rows
)


print(
    overall_df.to_string(
        index=False,
        float_format=lambda x:
        f"{x:.6f}"
    )
)


overall_df.to_csv(
    FINAL_DIR
    / "01_overall_before_after.csv",
    index=False
)


# ============================================================
# 17A. TRUE MULTICLASS ROC-AUC (MACRO OvR) — W0 AND WT
# ============================================================
# Uses the SAME current-run rows from final_df, and their original
# One-shot Domain prompts. Never substitutes hard predictions for scores.
# W0 is freshly loaded from the original, unedited model checkpoint.
# The edited current model (WT) is never restored or modified.

print("\n" + "=" * 100)
print("TRUE ROC-AUC — MACRO ONE-vs-REST (W0 / WT)")
print("=" * 100)

assert MODEL_OBJ is not None, "WT model is unavailable for ROC-AUC."
assert "tokenizer" in globals() and tokenizer is not None, "Tokenizer unavailable."
assert "sample_id" in final_df.columns and "true_label" in final_df.columns

roc_df = final_df.copy().reset_index(drop=True)
roc_df["sample_id"] = roc_df["sample_id"].astype(str).str.strip()
assert roc_df["sample_id"].is_unique, "Duplicate IDs in current run."

# Match prompts ONLY from the current evaluation, or from in-memory meta.
if "exact_messages_json" not in roc_df.columns:
    assert "meta" in globals() and isinstance(meta, pd.DataFrame), (
        "Missing exact_messages_json: current in-memory meta is required."
    )
    prompt_meta = meta.copy()
    prompt_meta.index = prompt_meta.index.astype(str).str.strip()
    assert prompt_meta.index.is_unique, "Duplicate IDs in meta."
    assert "exact_messages_json" in prompt_meta.columns, (
        "meta does not contain exact_messages_json."
    )
    roc_df["exact_messages_json"] = roc_df["sample_id"].map(
        prompt_meta["exact_messages_json"]
    )

assert roc_df["exact_messages_json"].notna().all(), (
    "Original prompts are missing for some current-run samples; "
    "ROC-AUC cannot be evaluated faithfully."
)

# The observed class set is used for the macro average. For a paper-wide
# macro over a fixed label ontology, all classes must be represented.
roc_labels = sorted(roc_df["true_label"].map(norm_label).unique().tolist())
assert len(roc_labels) >= 2, "ROC-AUC needs at least two classes."
assert "__wrong__" not in roc_labels, "Invalid ground-truth class encountered."
print("Current-run samples:", len(roc_df), "| Observed classes:", len(roc_labels))
print("Classes:", roc_labels)

# IMPORTANT: If the official dataset has additional absent classes,
# this is a macro over observed classes, not an official all-class macro.
roc_class_counts = roc_df["true_label"].map(norm_label).value_counts()
print("Class support:\n", roc_class_counts.to_string())


def _original_prompt_ids(messages_json):
    messages = json.loads(messages_json) if isinstance(messages_json, str) else messages_json
    if isinstance(messages, list):
        content = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        content = str(messages)
    ids = tokenizer(content, add_special_tokens=False, return_tensors="pt").input_ids
    assert ids.shape[1] > 0, "Empty source prompt."
    return ids


def _label_mean_loglikelihood(model_obj, prompt_ids, class_name):
    """Teacher-forced mean log p(label tokens | original prompt)."""
    device = next(model_obj.parameters()).device
    pids = prompt_ids.to(device)
    lids = tokenizer(
        class_name, add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(device)
    assert lids.shape[1] > 0, f"Empty tokenization for {class_name}"

    ids = torch.cat([pids, lids], dim=1)
    start = pids.shape[1] - 1
    length = lids.shape[1]
    with torch.inference_mode():
        logits = model_obj(input_ids=ids).logits[0, start:start + length].float()
        log_probs = torch.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(1, lids[0, :, None]).squeeze(1)
    return float(token_log_probs.mean().item())


def _get_score_matrix(model_obj, prompts, class_names, model_tag):
    model_obj.eval()
    rows = []
    for idx, prompt in enumerate(prompts, 1):
        rows.append([
            _label_mean_loglikelihood(model_obj, prompt, c)
            for c in class_names
        ])
        if idx % 10 == 0 or idx == len(prompts):
            print(f"{model_tag} label scores: {idx}/{len(prompts)}")
    scores = np.asarray(rows, dtype=np.float64)
    assert scores.shape == (len(prompts), len(class_names))
    assert np.isfinite(scores).all(), f"Nonfinite {model_tag} scores"
    # Softmax converts the score vector to class-normalized relative scores.
    # This is NOT claimed to be calibrated probability of free-form generation.
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    return probs


def _classwise_macro_ovr(truth, probs, class_names):
    auc_by_class = {}
    for j, c in enumerate(class_names):
        target = (np.asarray(truth) == c).astype(int)
        assert 0 < target.sum() < len(target), (
            f"Cannot compute class {c} OvR AUC: no positives or negatives."
        )
        auc_by_class[c] = float(roc_auc_score(target, probs[:, j]))
    return float(np.mean(list(auc_by_class.values()))), auc_by_class


roc_prompt_ids = [
    _original_prompt_ids(v) for v in roc_df["exact_messages_json"]
]
roc_y = roc_df["true_label"].map(norm_label).to_numpy()

# WT: evaluate the actual edited model currently in memory.
roc_probs_wt = _get_score_matrix(MODEL_OBJ, roc_prompt_ids, roc_labels, "WT")
ROC_AUC_WT, auc_wt_by_class = _classwise_macro_ovr(
    roc_y, roc_probs_wt, roc_labels
)
print(f"WT macro OvR ROC-AUC: {ROC_AUC_WT:.6f}")

# W0: load ORIGINAL, UNEDITED model separately. This does not touch WT.
original_model_name = globals().get("MODEL_NAME", "Qwen/Qwen2.5-3B-Instruct")
assert original_model_name != str(RUN_DIR / "FINAL_WT_MODEL"), (
    "W0 source must not be the edited WT checkpoint."
)
print("Loading original W0 model:", original_model_name)
roc_base_model = AutoModelForCausalLM.from_pretrained(
    original_model_name,
    torch_dtype="auto",
    low_cpu_mem_usage=True,
)
# Use the current WT model device where possible (GH200/CSC).
roc_base_model.to(next(MODEL_OBJ.parameters()).device)
roc_base_model.eval()
try:
    roc_probs_w0 = _get_score_matrix(
        roc_base_model, roc_prompt_ids, roc_labels, "W0"
    )
finally:
    del roc_base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ROC_AUC_W0, auc_w0_by_class = _classwise_macro_ovr(
    roc_y, roc_probs_w0, roc_labels
)
print(f"W0 macro OvR ROC-AUC: {ROC_AUC_W0:.6f}")
print(f"WT macro OvR ROC-AUC: {ROC_AUC_WT:.6f}")

roc_dir = RUN_DIR / "FINAL_TRUE_ROC"
roc_dir.mkdir(parents=True, exist_ok=True)
roc_scores_df = roc_df[["sample_id", "true_label"]].copy()
for j, c in enumerate(roc_labels):
    roc_scores_df[f"W0_score_{c}"] = roc_probs_w0[:, j]
    roc_scores_df[f"WT_score_{c}"] = roc_probs_wt[:, j]
roc_scores_df.to_csv(roc_dir / "HumAID_TRUE_ROC_SCORES.csv", index=False)

roc_classwise_df = pd.DataFrame([
    {
        "class": c,
        "support": int(roc_class_counts[c]),
        "AUC_W0": auc_w0_by_class[c],
        "AUC_WT": auc_wt_by_class[c],
    }
    for c in roc_labels
])
roc_classwise_df.to_csv(roc_dir / "HumAID_TRUE_ROC_PER_CLASS.csv", index=False)

roc_summary_df = pd.DataFrame([{
    "Dataset": "HumAID",
    "N": len(roc_df),
    "N_Observed_Classes": len(roc_labels),
    "ROC_AUC_W0": ROC_AUC_W0,
    "ROC_AUC_WT": ROC_AUC_WT,
    "Score_Method": "softmax(mean_teacher_forced_label_loglikelihood)",
    "Protocol": "original_exact_messages_json_one_shot_domain",
}])
roc_summary_df.to_csv(roc_dir / "HumAID_TRUE_ROC_SUMMARY.csv", index=False)
print("Saved ROC details:", roc_dir)
# Keep the overall comparison table synchronized with the master summary.
overall_df = pd.concat([overall_df, pd.DataFrame([{
    "Metric": "ROC_AUC_Macro_OvR",
    "Before_W0": ROC_AUC_W0,
    "After_WT": ROC_AUC_WT,
    "Delta": ROC_AUC_WT - ROC_AUC_W0,
}])], ignore_index=True)
overall_df.to_csv(FINAL_DIR / "01_overall_before_after.csv", index=False)

# ============================================================
# 18. MASTER SUMMARY — REQUESTED METRICS ONLY
# ============================================================

master = {
    "N": len(final_df),
    "Accuracy_W0": score_w0["Accuracy"],
    "Accuracy_WT": score_wt["Accuracy"],
    "Precision_W0": score_w0["Precision_Macro"],
    "Precision_WT": score_wt["Precision_Macro"],
    "Recall_W0": score_w0["Recall_Macro"],
    "Recall_WT": score_wt["Recall_Macro"],
    "MacroF1_W0": score_w0["Macro_F1"],
    "MacroF1_WT": score_wt["Macro_F1"],
    "MCC_W0": score_w0["MCC"],
    "MCC_WT": score_wt["MCC"],
    "ROC_AUC_W0": ROC_AUC_W0,
    "ROC_AUC_WT": ROC_AUC_WT,
}

master_df = pd.DataFrame([master])
master_path = FINAL_DIR / "00_MASTER_CURRENT_RUN.csv"
master_df.to_csv(master_path, index=False)

print("\n" + "=" * 100)
print("MASTER CURRENT RUN — REQUESTED METRICS ONLY")
print("=" * 100)
print(master_df.to_string(index=False))
print("\nSaved:", master_path)
print("Overall metrics:", FINAL_DIR / "01_overall_before_after.csv")
print("ROC-AUC details:", roc_dir)
print("\nIMPORTANT: model is still at WT. Do not restore W0 until checkpoint is verified.")


FINAL HARVEST — CURRENT WT
RUN_DIR  : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run
SAVE_DIR : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/FINAL_ALL_METRICS

1. SAVING CURRENT WT MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] WT model saved from variable `model`
/scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/FINAL_WT_MODEL
[OK] tokenizer saved

2. LOADING FINAL W0 -> WT RESULTS
[OK] Loaded current run predictions: /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HumAID_FINAL_W0_WT_generated.csv
Evaluation rows: 500

3. OVERALL TASK PERFORMANCE
         Metric  Before_W0  After_WT    Delta
       Accuracy   0.536000  0.618000 0.082000
Precision_Macro   0.478598  0.575227 0.096629
   Recall_Macro   0.438680  0.494670 0.055990
       Macro_F1   0.413552  0.438761 0.025209
            MCC   0.472145  0.548219 0.076074

TRUE ROC-AUC — MACRO ONE-vs-REST (W0 / WT)
Current-run samples: 500 | Observed classes: 10
Classes: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

W0 label scores: 10/500
W0 label scores: 20/500
W0 label scores: 30/500
W0 label scores: 40/500
W0 label scores: 50/500
W0 label scores: 60/500
W0 label scores: 70/500
W0 label scores: 80/500
W0 label scores: 90/500
W0 label scores: 100/500
W0 label scores: 110/500
W0 label scores: 120/500
W0 label scores: 130/500
W0 label scores: 140/500
W0 label scores: 150/500
W0 label scores: 160/500
W0 label scores: 170/500
W0 label scores: 180/500
W0 label scores: 190/500
W0 label scores: 200/500
W0 label scores: 210/500
W0 label scores: 220/500
W0 label scores: 230/500
W0 label scores: 240/500
W0 label scores: 250/500
W0 label scores: 260/500
W0 label scores: 270/500
W0 label scores: 280/500
W0 label scores: 290/500
W0 label scores: 300/500
W0 label scores: 310/500
W0 label scores: 320/500
W0 label scores: 330/500
W0 label scores: 340/500
W0 label scores: 350/500
W0 label scores: 360/500
W0 label scores: 370/500
W0 label scores: 380/500
W0 label scores: 390/500
W0 label scores: 400/500
W0 label 

In [68]:
# ============================================================
# FLUENCY / GENERATION ENTROPY (GE) — ALL 500 HumAID SAMPLES
# Current WT model
# Does NOT modify model weights
# ============================================================

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch


# ============================================================
# 0. Paths
# ============================================================

assert "RUN_DIR" in globals(), "Run Global Configuration first."
RUN_DIR = Path(RUN_DIR).resolve()
assert RUN_DIR.is_dir(), f"Run directory missing: {RUN_DIR}"

OUT_DIR = (
    RUN_DIR
    / "FINAL_COMPLETE_METRICS"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Current WT model
# ============================================================

MODEL = None

for name in [
    "model",
    "model_edited",
    "edited_model",
]:

    if (
        name in globals()
        and globals()[name] is not None
    ):

        MODEL = globals()[name]

        print("Using model:", name)
        break


assert MODEL is not None, (
    "No WT model found in memory."
)

assert (
    "tokenizer" in globals()
    and tokenizer is not None
), (
    "Tokenizer not found."
)


MODEL.eval()

DEVICE = next(
    MODEL.parameters()
).device

print("Device:", DEVICE)


# ============================================================
# 2. Load all 500 evaluation samples
# ============================================================

# Read only the current run's completed W0/WT evaluation.
_csv_candidates = [
    RUN_DIR / "HumAID_FINAL_W0_WT_generated.csv",
    RUN_DIR / "HUMAID_FINAL_W0_WT_generated.csv",
]
_existing_csvs = [p for p in _csv_candidates if p.is_file()]
if len(_existing_csvs) != 1:
    raise FileNotFoundError(
        f"Expected exactly one current-run W0/WT evaluation CSV; found {_existing_csvs}. "
        f"Checked: {_csv_candidates}"
    )
FINAL_FILE = _existing_csvs[0]
print("Reading current-run predictions:", FINAL_FILE)


flu_df = pd.read_csv(FINAL_FILE)


assert "text" in flu_df.columns
assert "sample_id" in flu_df.columns


flu_eval = (
    flu_df[
        [
            "sample_id",
            "text"
        ]
    ]
    .drop_duplicates(
        subset=["sample_id"],
        keep="first"
    )
    .reset_index(drop=True)
)


print(
    "Unique evaluation samples:",
    len(flu_eval)
)


assert len(flu_eval) == 500, (
    f"Expected 500 unique samples, "
    f"but found {len(flu_eval)}."
)


# ============================================================
# 3. Free-generation prompt
# ============================================================

def build_fluency_prompt(text):

    return (
        "Read the following disaster-related social media post "
        "and briefly explain its content in natural language.\n\n"
        f"Post: {text}\n\n"
        "Explanation:"
    )


# ============================================================
# 4. Generate on ALL 500 samples
# ============================================================

generated_rows = []


for idx, row in flu_eval.iterrows():

    prompt = build_fluency_prompt(
        row["text"]
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )


    inputs = {
        k: v.to(DEVICE)
        for k, v
        in inputs.items()
    }


    input_len = (
        inputs["input_ids"]
        .shape[1]
    )


    with torch.inference_mode():

        output = MODEL.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )


    new_tokens = output[
        0,
        input_len:
    ]


    generated_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


    generated_rows.append(
        {
            "sample_id":
                row["sample_id"],

            "source_text":
                row["text"],

            "generated_text_WT":
                generated_text,
        }
    )


    done = idx + 1

    if (
        done % 10 == 0
        or
        done == len(flu_eval)
    ):

        print(
            f"Generated "
            f"{done}/"
            f"{len(flu_eval)}"
        )


gen_df = pd.DataFrame(
    generated_rows
)


# ============================================================
# 5. N-gram entropy
# ============================================================

def ngram_entropy(
    text,
    n
):

    tokens = (
        str(text)
        .strip()
        .split()
    )


    if len(tokens) < n:
        return np.nan


    grams = [
        tuple(
            tokens[i:i+n]
        )
        for i in range(
            len(tokens) - n + 1
        )
    ]


    counts = Counter(
        grams
    )


    probs = np.array(
        list(
            counts.values()
        ),
        dtype=float
    )


    probs = (
        probs
        /
        probs.sum()
    )


    return float(
        -np.sum(
            probs
            *
            np.log2(
                probs + 1e-12
            )
        )
    )


# ============================================================
# 6. Weighted GE
# ============================================================

def generation_entropy(text):

    e2 = ngram_entropy(
        text,
        2
    )

    e3 = ngram_entropy(
        text,
        3
    )


    if (
        not np.isfinite(e2)
        or
        not np.isfinite(e3)
    ):

        return np.nan


    return (
        (2.0 / 3.0) * e2
        +
        (4.0 / 3.0) * e3
    )


gen_df[
    "GE"
] = (
    gen_df[
        "generated_text_WT"
    ]
    .map(
        generation_entropy
    )
)


# ============================================================
# 7. Final GE over all 500
# ============================================================

valid_ge = (
    gen_df[
        "GE"
    ]
    .dropna()
)


assert len(valid_ge) > 0


FLUENCY_GE = float(
    valid_ge.mean()
)


print()
print("=" * 80)
print("FLUENCY / GENERATION ENTROPY — ALL 500")
print("=" * 80)

print(
    "Total samples:",
    len(gen_df)
)

print(
    "Valid GE samples:",
    len(valid_ge)
)

print(
    f"Fluency_GE WT = "
    f"{FLUENCY_GE:.6f}"
)

print(
    "Mean generated length:",
    gen_df[
        "generated_text_WT"
    ]
    .str.split()
    .map(len)
    .mean()
)


# ============================================================
# 8. Save
# ============================================================

DETAIL_PATH = (
    OUT_DIR
    / "FLUENCY_WT_GENERATIONS_500.csv"
)

SUMMARY_PATH = (
    OUT_DIR
    / "FLUENCY_GE_500.csv"
)


gen_df.to_csv(
    DETAIL_PATH,
    index=False
)


pd.DataFrame(
    [
        {
            "Dataset":
                "HumAID",

            "N":
                len(gen_df),

            "Valid_GE_N":
                len(valid_ge),

            "Fluency_GE":
                FLUENCY_GE,
        }
    ]
).to_csv(
    SUMMARY_PATH,
    index=False
)


print()
print("Saved:")
print(DETAIL_PATH)
print(SUMMARY_PATH)

Using model: model
Device: cuda:0
Reading current-run predictions: /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HumAID_FINAL_W0_WT_generated.csv
Unique evaluation samples: 500
Generated 10/500
Generated 20/500
Generated 30/500
Generated 40/500
Generated 50/500
Generated 60/500
Generated 70/500
Generated 80/500
Generated 90/500
Generated 100/500
Generated 110/500
Generated 120/500
Generated 130/500
Generated 140/500
Generated 150/500
Generated 160/500
Generated 170/500
Generated 180/500
Generated 190/500
Generated 200/500
Generated 210/500
Generated 220/500
Generated 230/500
Generated 240/500
Generated 250/500
Generated 260/500
Generated 270/500
Generated 280/500
Generated 290/500
Generated 300/500
Generated 310/500
Generated 320/500
Generated 330/500
Generated 340/500
Generated 350/500
Generated 360/500
Generated 370/500
Generated 380/500
Generated 390/500
Generated 400/500
Generated 410/500
Generated 420/500
Generated 430/500
Generated 440/500
Generated 450/500
Genera

In [69]:
# ============================================================
# GENERALIZATION + PORTABILITY + CONSISTENCY + SCORE
# HumAID — CURRENT WT MODEL
#
# Does NOT modify model weights
#
# Evaluation:
#   ES = Efficacy
#   PS = Paraphrase Generalization
#   NS = Specificity
#   Portability = robustness to classification prompt variants
#   RS = classification consistency across paraphrases
#   S  = Harmonic Mean(ES, PS, NS)
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch


# ============================================================
# 0. PATHS
# ============================================================

assert "RUN_DIR" in globals(), "Run Global Configuration first."
RUN_DIR = Path(RUN_DIR).resolve()
assert RUN_DIR.is_dir(), f"Run directory missing: {RUN_DIR}"

OUT_DIR = (
    RUN_DIR
    / "FINAL_COMPLETE_METRICS"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Read only the current run's completed W0/WT evaluation.
_csv_candidates = [
    RUN_DIR / "HumAID_FINAL_W0_WT_generated.csv",
    RUN_DIR / "HUMAID_FINAL_W0_WT_generated.csv",
]
_existing_csvs = [p for p in _csv_candidates if p.is_file()]
if len(_existing_csvs) != 1:
    raise FileNotFoundError(
        f"Expected exactly one current-run W0/WT evaluation CSV; found {_existing_csvs}. "
        f"Checked: {_csv_candidates}"
    )
FINAL_FILE = _existing_csvs[0]
print("Reading current-run predictions:", FINAL_FILE)


# ============================================================
# 1. GET CURRENT WT MODEL
# ============================================================

MODEL = None
MODEL_VAR = None

for name in [
    "model",
    "model_edited",
    "edited_model",
]:

    if (
        name in globals()
        and globals()[name] is not None
    ):

        MODEL = globals()[name]
        MODEL_VAR = name
        break


assert MODEL is not None, (
    "Current WT model not found."
)

assert (
    "tokenizer" in globals()
    and tokenizer is not None
), (
    "Tokenizer not found."
)


MODEL.eval()

DEVICE = next(
    MODEL.parameters()
).device


print("=" * 90)
print("GENERALIZATION / PORTABILITY / CONSISTENCY")
print("=" * 90)

print("Model variable:", MODEL_VAR)
print("Device:", DEVICE)


# ============================================================
# 2. LOAD FINAL W0 -> WT RESULTS
# ============================================================

eval_df = pd.read_csv(FINAL_FILE)


print(
    "Evaluation samples:",
    len(eval_df)
)


# ============================================================
# 3. NORMALIZE LABEL
# ============================================================

def norm_label(x):

    if pd.isna(x):
        return "__wrong__"

    x = str(x).strip().lower()

    x = re.sub(
        r"[\s\-]+",
        "_",
        x
    )

    x = re.sub(
        r"[^a-z0-9_]",
        "",
        x
    )

    x = re.sub(
        r"_+",
        "_",
        x
    ).strip("_")

    return (
        x
        if x
        else "__wrong__"
    )


# ============================================================
# 4. W0 COLUMN
# ============================================================

if "predicted_label_W0" in eval_df.columns:

    W0_COL = "predicted_label_W0"

elif "predicted_label" in eval_df.columns:

    W0_COL = "predicted_label"

else:

    raise KeyError(
        "W0 prediction column not found."
    )


# ============================================================
# 5. LABEL SET
# ============================================================

labels = sorted(
    eval_df[
        "true_label"
    ]
    .map(norm_label)
    .unique()
    .tolist()
)


print("\nLabels:")

for x in labels:
    print(" -", x)


label_text = ", ".join(
    labels
)


# ============================================================
# 6. FIND EDIT TARGETS
#
# Original W0-wrong examples
# ============================================================

true_norm = (
    eval_df[
        "true_label"
    ]
    .map(norm_label)
)


w0_norm = (
    eval_df[
        W0_COL
    ]
    .map(norm_label)
)


target_mask = (
    true_norm
    !=
    w0_norm
)


targets = (
    eval_df[
        target_mask
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "\nEdit targets:",
    len(targets)
)


assert len(targets) > 0


# ============================================================
# 7. GENERATION HELPER
# ============================================================

def generate_text(
    prompt,
    max_new_tokens=80
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v
        in inputs.items()
    }

    input_len = (
        inputs[
            "input_ids"
        ]
        .shape[1]
    )


    with torch.inference_mode():

        output = MODEL.generate(
            **inputs,

            max_new_tokens=
                max_new_tokens,

            do_sample=False,

            pad_token_id=
                tokenizer.eos_token_id,
        )


    new_tokens = (
        output[
            0,
            input_len:
        ]
    )


    return (
        tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        )
        .strip()
    )


# ============================================================
# 8. PARAPHRASE GENERATION
#
# 3 paraphrases per edit target
# ============================================================

N_PARAPHRASES = 3


def generate_paraphrases(
    text
):

    prompt = f"""
Paraphrase the disaster-related social media post below in
{N_PARAPHRASES} different ways.

Rules:
- Preserve exactly the same factual meaning.
- Do not add new information.
- Do not remove important information.
- Do not mention classification labels.
- Return exactly {N_PARAPHRASES} lines.
- Put one paraphrase on each line.
- Do not number the lines.

Original post:
{text}

Paraphrases:
""".strip()


    output = generate_text(
        prompt,
        max_new_tokens=180
    )


    lines = [
        x.strip()
        for x in output.splitlines()
        if x.strip()
    ]


    # remove accidental numbering
    cleaned = []

    for x in lines:

        x = re.sub(
            r"^\s*[\-\*\d\.\)\:]+\s*",
            "",
            x
        ).strip()

        if x:
            cleaned.append(x)


    # unique
    unique = []

    for x in cleaned:

        if x not in unique:
            unique.append(x)


    return unique[
        :N_PARAPHRASES
    ]


# ============================================================
# 9. CLASSIFICATION HELPER
# ============================================================

def classify_text(
    text,
    prompt_variant=0
):

    if prompt_variant == 0:

        prompt = f"""
Classify the following disaster-related social media post.

Allowed labels:
{label_text}

Return exactly ONE label from the allowed labels.
Do not provide an explanation.

Post:
{text}

Label:
""".strip()


    elif prompt_variant == 1:

        prompt = f"""
Which category best describes this disaster-related post?

Categories:
{label_text}

Respond only with the exact category name.

Text:
{text}

Category:
""".strip()


    else:

        prompt = f"""
Assign one disaster-response class to the text below.

Possible classes:
{label_text}

Output only one class name.

Input:
{text}

Answer:
""".strip()


    raw = generate_text(
        prompt,
        max_new_tokens=24
    )


    pred = norm_label(
        raw.splitlines()[0]
        if raw
        else "__wrong__"
    )


    # --------------------------------------------------------
    # Sometimes model adds a few words.
    # Match a valid label if one appears.
    # --------------------------------------------------------

    if pred not in labels:

        raw_norm = norm_label(
            raw
        )

        matches = [
            label
            for label in labels
            if label in raw_norm
        ]

        if len(matches) == 1:

            pred = matches[0]

        else:

            pred = "__wrong__"


    return pred, raw


# ============================================================
# 10. RUN PARAPHRASE GENERALIZATION
# ============================================================

rows = []


for idx, row in targets.iterrows():

    sample_id = row[
        "sample_id"
    ]

    original_text = str(
        row["text"]
    )

    true_label = norm_label(
        row["true_label"]
    )


    paraphrases = (
        generate_paraphrases(
            original_text
        )
    )


    # --------------------------------------------------------
    # If fewer than 3 were produced,
    # keep whatever was produced.
    # --------------------------------------------------------

    for p_idx, para in enumerate(
        paraphrases
    ):

        pred, raw = classify_text(
            para,
            prompt_variant=0
        )


        rows.append(
            {
                "sample_id":
                    sample_id,

                "true_label":
                    true_label,

                "paraphrase_id":
                    p_idx + 1,

                "original_text":
                    original_text,

                "paraphrase":
                    para,

                "prediction_WT":
                    pred,

                "correct":
                    pred
                    ==
                    true_label,

                "raw_prediction":
                    raw,
            }
        )


    done = idx + 1

    if (
        done % 10 == 0
        or
        done == len(targets)
    ):

        print(
            f"Paraphrase evaluation: "
            f"{done}/{len(targets)}"
        )


para_df = pd.DataFrame(
    rows
)


assert len(para_df) > 0, (
    "No paraphrases generated."
)


# ============================================================
# 11. GENERALIZATION / PS
# ============================================================

PS = float(
    para_df[
        "correct"
    ]
    .mean()
)


print()
print("=" * 90)
print("GENERALIZATION")
print("=" * 90)

print(
    "Paraphrase probes:",
    len(para_df)
)

print(
    f"Generalization / PS = "
    f"{PS:.6f}"
)


# ============================================================
# 12. CLASSIFICATION CONSISTENCY / RS
#
# For each edit target:
# how consistently do its paraphrases receive the same label?
#
# We use majority-label agreement.
# ============================================================

consistency_rows = []


for sid, group in para_df.groupby(
    "sample_id"
):

    preds = (
        group[
            "prediction_WT"
        ]
        .tolist()
    )


    if not preds:
        continue


    counts = (
        pd.Series(preds)
        .value_counts()
    )


    majority_n = int(
        counts.iloc[0]
    )


    consistency = (
        majority_n
        /
        len(preds)
    )


    consistency_rows.append(
        {
            "sample_id":
                sid,

            "n_paraphrases":
                len(preds),

            "Consistency":
                consistency,
        }
    )


consistency_df = pd.DataFrame(
    consistency_rows
)


RS = float(
    consistency_df[
        "Consistency"
    ]
    .mean()
)


print()
print("=" * 90)
print("CONSISTENCY")
print("=" * 90)

print(
    f"Consistency / RS = "
    f"{RS:.6f}"
)


# ============================================================
# 13. PORTABILITY
#
# Classification robustness to alternative task instructions.
#
# Same target text; 3 different instruction formulations.
# ============================================================

portability_rows = []


for idx, row in targets.iterrows():

    text = str(
        row["text"]
    )

    true_label = norm_label(
        row["true_label"]
    )


    for variant in range(3):

        pred, raw = classify_text(
            text,
            prompt_variant=variant
        )


        portability_rows.append(
            {
                "sample_id":
                    row["sample_id"],

                "prompt_variant":
                    variant,

                "true_label":
                    true_label,

                "prediction":
                    pred,

                "correct":
                    pred
                    ==
                    true_label,

                "raw_prediction":
                    raw,
            }
        )


    done = idx + 1

    if (
        done % 20 == 0
        or
        done == len(targets)
    ):

        print(
            f"Portability: "
            f"{done}/{len(targets)}"
        )


port_df = pd.DataFrame(
    portability_rows
)


PORTABILITY = float(
    port_df[
        "correct"
    ]
    .mean()
)


print()
print("=" * 90)
print("PORTABILITY")
print("=" * 90)

print(
    f"Portability = "
    f"{PORTABILITY:.6f}"
)


# ============================================================
# 14. EFFICACY / ES
# ============================================================

wt_norm = (
    eval_df[
        "predicted_label_WT"
    ]
    .map(norm_label)
)


wrong_w0 = (
    true_norm
    !=
    w0_norm
)


ES = float(
    (
        wt_norm[
            wrong_w0
        ]
        ==
        true_norm[
            wrong_w0
        ]
    )
    .mean()
)


# ============================================================
# 15. SPECIFICITY / NS
# ============================================================

correct_w0 = (
    true_norm
    ==
    w0_norm
)


NS = float(
    (
        wt_norm[
            correct_w0
        ]
        ==
        true_norm[
            correct_w0
        ]
    )
    .mean()
)


# ============================================================
# 16. SCORE S
#
# Harmonic mean ES / PS / NS
# ============================================================

if (
    ES > 0
    and
    PS > 0
    and
    NS > 0
):

    SCORE_S = (
        3.0
        /
        (
            (1.0 / ES)
            +
            (1.0 / PS)
            +
            (1.0 / NS)
        )
    )

else:

    SCORE_S = np.nan


print()
print("=" * 90)
print("MAIN EDITING METRICS")
print("=" * 90)

print(
    f"Efficacy ES       = {ES:.6f}"
)

print(
    f"Generalization PS = {PS:.6f}"
)

print(
    f"Specificity NS    = {NS:.6f}"
)

print(
    f"Score S           = {SCORE_S:.6f}"
)

print(
    f"Consistency RS    = {RS:.6f}"
)

print(
    f"Portability       = {PORTABILITY:.6f}"
)


# ============================================================
# 17. SAVE
# ============================================================

para_df.to_csv(
    OUT_DIR
    / "GENERALIZATION_PARAPHRASE_DETAILS.csv",
    index=False
)


consistency_df.to_csv(
    OUT_DIR
    / "CONSISTENCY_DETAILS.csv",
    index=False
)


port_df.to_csv(
    OUT_DIR
    / "PORTABILITY_DETAILS.csv",
    index=False
)


summary = pd.DataFrame(
    [
        {
            "Dataset":
                "HumAID",

            "Efficacy_ES":
                ES,

            "Generalization_PS":
                PS,

            "Specificity_NS":
                NS,

            "Score_S":
                SCORE_S,

            "Consistency_RS":
                RS,

            "Portability":
                PORTABILITY,

            "N_Edit_Targets":
                len(targets),

            "N_Paraphrase_Probes":
                len(para_df),
        }
    ]
)


summary.to_csv(
    OUT_DIR
    / "GENERALIZATION_CONSISTENCY_PORTABILITY_SUMMARY.csv",
    index=False
)


print()
print("Saved to:")
print(OUT_DIR)

Reading current-run predictions: /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HumAID_FINAL_W0_WT_generated.csv
GENERALIZATION / PORTABILITY / CONSISTENCY
Model variable: model
Device: cuda:0
Evaluation samples: 500

Labels:
 - caution_and_advice
 - displaced_people_and_evacuations
 - infrastructure_and_utility_damage
 - injured_or_dead_people
 - missing_or_found_people
 - not_humanitarian
 - other_relevant_information
 - requests_or_urgent_needs
 - rescue_volunteering_or_donation_effort
 - sympathy_and_support

Edit targets: 232
Paraphrase evaluation: 10/232
Paraphrase evaluation: 20/232
Paraphrase evaluation: 30/232
Paraphrase evaluation: 40/232
Paraphrase evaluation: 50/232
Paraphrase evaluation: 60/232
Paraphrase evaluation: 70/232
Paraphrase evaluation: 80/232
Paraphrase evaluation: 90/232
Paraphrase evaluation: 100/232
Paraphrase evaluation: 110/232
Paraphrase evaluation: 120/232
Paraphrase evaluation: 130/232
Paraphrase evaluation: 140/232
Paraphrase evaluation: 

In [ ]:
# ============================================================
# FINAL COMPLETE METHOD-SPECIFIC EVALUATION — HumAID
#
# Main method metrics:
#   ES  = Efficacy
#   RG  = Ripple Generalization
#   NS  = Protect Specificity
#   Ret = Previous-edit Retention
#   FR  = Forgetting Rate
#   S   = Harmonic Mean(ES, RG, NS)
#
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. CONFIG — CURRENT RUN PATHS
# ============================================================

assert "RUN_DIR" in globals(), "Run Global Configuration first."
RUN_DIR = Path(RUN_DIR).resolve()
EXPECTED_N = int(N_SAMPLES) if "N_SAMPLES" in globals() else None
OUT_DIR = RUN_DIR / "FINAL_CORRECT_METRICS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. FILE FINDER — CURRENT RUN ONLY
# ============================================================

def find_run_file(filename, required=False):
    matches = sorted(
        p for p in RUN_DIR.rglob("*")
        if p.is_file() and p.name.casefold() == filename.casefold()
    )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple matches for {filename}:\n"
            + "\n".join(map(str, matches))
        )

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Missing file: {filename}\nRUN_DIR: {RUN_DIR}"
            )
        return None

    return matches[0]


FINAL_FILE = find_run_file(
    "HumAID_FINAL_W0_WT_generated.csv", required=True
)

RIPPLE_FILE = find_run_file(
    "HumAID_ripple_sets.csv", required=True
)

PROTECT_FILE = find_run_file(
    "HumAID_protect_sets.csv", required=True
)

SEQ_INSTANCE_FILE = find_run_file(
    "HumAID_seq_edit_generated_per_instance.csv", required=True
)

SEQ_DIAG_FILE = find_run_file(
    "HumAID_seq_edit_generated_diagnostic.csv", required=True
)

PARA_FILE = find_run_file(
    "GENERALIZATION_PARAPHRASE_DETAILS.csv"
)

FLUENCY_FILE = (
    find_run_file(f"FLUENCY_GE_{EXPECTED_N}.csv")
    if EXPECTED_N is not None else None
)

# Always load evaluation from the selected run's CSV,
# not from a potentially stale notebook variable.
EVAL_FROM_MEMORY = None

print("\nCURRENT RUN:", RUN_DIR)

for name, path in {
    "Final predictions": FINAL_FILE,
    "Ripple sets": RIPPLE_FILE,
    "Protect sets": PROTECT_FILE,
    "Sequential instances": SEQ_INSTANCE_FILE,
    "Sequential diagnostics": SEQ_DIAG_FILE,
    "Paraphrase": PARA_FILE,
    "Fluency": FLUENCY_FILE,
}.items():
    print(name, ":", path)
# ============================================================
# 2. HELPERS
# ============================================================

def norm_label(x):

    if pd.isna(x):
        return "__wrong__"

    x = str(x).strip().lower()
    x = re.sub(r"[\s\-]+", "_", x)
    x = re.sub(r"[^a-z0-9_]", "", x)
    x = re.sub(r"_+", "_", x).strip("_")

    return x if x else "__wrong__"


def safe_mean(x):

    x = pd.to_numeric(x, errors="coerce").dropna()

    if len(x) == 0:
        return np.nan

    return float(x.mean())


# ============================================================
# 3. LOAD CURRENT EVALUATION (NO 500-ROW OVERWRITE)
# ============================================================

if EVAL_FROM_MEMORY is not None:
    df = EVAL_FROM_MEMORY.copy()
else:
    df = pd.read_csv(FINAL_FILE).copy()
df["sample_id"] = df["sample_id"].astype(str)

assert EXPECTED_N is None or len(df) == EXPECTED_N, (
    f"Expected {EXPECTED_N} rows, got {len(df)}"
)
print("Evaluation samples:", len(df))

if "predicted_label_W0" in df.columns:
    W0_COL = "predicted_label_W0"
elif "predicted_label" in df.columns:
    W0_COL = "predicted_label"
else:
    raise KeyError("W0 prediction column missing")

TRUE = df["true_label"].map(norm_label)
W0 = df[W0_COL].map(norm_label)
WT = df["predicted_label_WT"].map(norm_label)

correct_W0 = W0.to_numpy() == TRUE.to_numpy()
correct_WT = WT.to_numpy() == TRUE.to_numpy()


# ============================================================
# 4. CLASS LABELS
# ============================================================

raw_labels = (
    df[["true_label"]]
    .drop_duplicates()
    ["true_label"]
    .astype(str)
    .tolist()
)

# normalized -> original textual label
label_lookup = {norm_label(x): x for x in raw_labels}

class_names = sorted(label_lookup.keys())

print("\nClasses:", len(class_names))

for c in class_names:
    print(c, "->", label_lookup[c])


# ============================================================
# 5. EDIT TARGETS
# ============================================================

ripple_sets = pd.read_csv(RIPPLE_FILE)
ripple_sets["sample_id"] = ripple_sets["sample_id"].astype(str)

EDIT_IDS = set(ripple_sets["sample_id"])

edit_mask = df["sample_id"].isin(EDIT_IDS).to_numpy()

N_EDIT = int(edit_mask.sum())


# ============================================================
# 6. EFFICACY
# ============================================================

ES_BEFORE = float(correct_W0[edit_mask].mean())
ES = float(correct_WT[edit_mask].mean())


# ============================================================
# 7. RIPPLE GENERALIZATION — SAMPLE LEVEL
# ============================================================

seq_inst = pd.read_csv(SEQ_INSTANCE_FILE).copy()

ripple_inst = seq_inst[
    seq_inst["set_type"].astype(str).str.lower().str.contains("ripple")
].copy()

dedup_cols = [
    c for c in ["t", "group_id", "sample_id"]
    if c in ripple_inst.columns
]

if dedup_cols:
    ripple_inst = ripple_inst.drop_duplicates(dedup_cols, keep="last")

rb = pd.to_numeric(ripple_inst["correct_before"], errors="coerce")
ra = pd.to_numeric(ripple_inst["correct_after"], errors="coerce")

valid = rb.notna() & ra.notna()

rb = rb[valid]
ra = ra[valid]

RG_BEFORE = float(rb.mean())
RG_AFTER = float(ra.mean())
RG_GAIN = RG_AFTER - RG_BEFORE

N_RIPPLE_EVAL = len(ra)


# ============================================================
# 8. PROTECT / SPECIFICITY
# ============================================================

protect = pd.read_csv(PROTECT_FILE)
protect["sample_id"] = protect["sample_id"].astype(str)

PROTECT_IDS = set(protect["sample_id"])

protect_mask = df["sample_id"].isin(PROTECT_IDS).to_numpy()

N_PROTECT = int(protect_mask.sum())

PROTECT_ACC_W0 = float(correct_W0[protect_mask].mean())
PROTECT_ACC_WT = float(correct_WT[protect_mask].mean())

# Method-specific specificity
NS = PROTECT_ACC_WT

# Preservation:
# among protect examples correct at W0,
# fraction still correct at WT
preserve_mask = protect_mask & correct_W0

PRESERVATION = float(correct_WT[preserve_mask].mean())

DAMAGE_RATE = 1.0 - PRESERVATION


# ============================================================
# 9. SEQUENTIAL RETENTION
# ============================================================

seq_diag = pd.read_csv(SEQ_DIAG_FILE).copy()

seq_diag["_t"] = pd.to_numeric(seq_diag["t"], errors="coerce")

seq_diag = seq_diag.sort_values("_t").reset_index(drop=True)

ret_top1 = pd.to_numeric(seq_diag["prev_retention_teacher_top1"], errors="coerce").dropna()

RETENTION_FINAL = float(ret_top1.iloc[-1])
RETENTION_MEAN = float(ret_top1.mean())

FORGETTING_RATE = 1.0 - RETENTION_FINAL

ret_exact = pd.to_numeric(seq_diag["prev_retention_teacher_exact"], errors="coerce").dropna()

RETENTION_EXACT_FINAL = float(ret_exact.iloc[-1]) if len(ret_exact) else np.nan


# ============================================================
# 10. PROTECT KL
# ============================================================

PROTECT_KL = safe_mean(seq_diag["protect_kl"])


# ============================================================
# 11. COMPOSITE SCORE
#
# Harmonic mean:
#
# ES + Ripple Generalization + Specificity
# ============================================================

if ES > 0 and RG_AFTER > 0 and NS > 0:
    SCORE_S = 3.0 / (1.0 / ES + 1.0 / RG_AFTER + 1.0 / NS)
else:
    SCORE_S = np.nan


# ============================================================
# 21. PARAPHRASE ROBUSTNESS — AUXILIARY
# ============================================================

PARAPHRASE_ROBUSTNESS = np.nan
PARAPHRASE_CONSISTENCY = np.nan
N_PARAPHRASE = 0

if PARA_FILE is not None and PARA_FILE.exists():

    para = pd.read_csv(PARA_FILE).copy()

    if "correct" in para.columns:

        c = (
            para["correct"]
            .astype(str)
            .str.lower()
            .map({"true": 1, "false": 0, "1": 1, "0": 0})
            .dropna()
        )

        if len(c):
            PARAPHRASE_ROBUSTNESS = float(c.mean())
            N_PARAPHRASE = len(c)

    if "sample_id" in para.columns and "prediction_WT" in para.columns:

        consistency = []

        for sid, g in para.groupby("sample_id"):
            preds = g["prediction_WT"].map(norm_label)
            if len(preds):
                vc = preds.value_counts()
                consistency.append(float(vc.iloc[0] / len(preds)))

        if consistency:
            PARAPHRASE_CONSISTENCY = float(np.mean(consistency))


# ============================================================
# 22. FLUENCY
# ============================================================

FLUENCY_GE = np.nan
FLUENCY_N = np.nan

if FLUENCY_FILE is not None and FLUENCY_FILE.exists():

    flu = pd.read_csv(FLUENCY_FILE)

    for c in ["Fluency_GE", "GE"]:
        if c in flu.columns:
            vals = pd.to_numeric(flu[c], errors="coerce").dropna()
            if len(vals):
                FLUENCY_GE = float(vals.iloc[-1])
                break

    if "N" in flu.columns:
        vals = pd.to_numeric(flu["N"], errors="coerce").dropna()
        if len(vals):
            FLUENCY_N = int(vals.iloc[-1])


# ============================================================
# 23. LOCALIZATION SUMMARY
# ============================================================

LOCALIZATION = {
    "Mean_L_edit":
        safe_mean(seq_diag["L_edit"]) if "L_edit" in seq_diag.columns else np.nan,
    "Mean_L_update":
        safe_mean(seq_diag["L_update"]) if "L_update" in seq_diag.columns else np.nan,
    "Mean_Update_Norm":
        safe_mean(seq_diag["update_norm"]) if "update_norm" in seq_diag.columns else np.nan,
}


# ============================================================
# 24. SEQUENTIAL SCALABILITY
# ============================================================

scalability = seq_diag.copy()

if "n_members" in scalability.columns:
    scalability["Cumulative_Edits"] = (
        pd.to_numeric(scalability["n_members"], errors="coerce")
        .fillna(0)
        .cumsum()
    )


# ============================================================
# 25. COVERAGE CHECK
# ============================================================

ALL_IDS = set(df["sample_id"])

UNASSIGNED_IDS = ALL_IDS - (EDIT_IDS | PROTECT_IDS)


# ============================================================
# 26. FINAL MAIN METHOD TABLE
# ============================================================

MAIN = pd.DataFrame([
    {
        "Dataset":                "HumAID",
        "N_Total":                len(df),
        "N_Edit_Targets":         N_EDIT,
        "N_Ripple_Evaluations":   N_RIPPLE_EVAL,
        "N_Protect":              N_PROTECT,
        "N_Unassigned":           len(UNASSIGNED_IDS),

        # --------------------
        # Core editing metrics
        # --------------------
        "Score_S":                SCORE_S,
        "Efficacy_ES":            ES,
        "Ripple_Generalization":  RG_AFTER,
        "Ripple_Before":          RG_BEFORE,
        "Ripple_Gain":            RG_GAIN,
        "Specificity_NS":         NS,
        "Preservation":           PRESERVATION,
        "Damage_Rate":            DAMAGE_RATE,
        "Retention_Final":        RETENTION_FINAL,
        "Retention_Mean":         RETENTION_MEAN,
        "Retention_Exact_Final":  RETENTION_EXACT_FINAL,
        "Forgetting_Rate":        FORGETTING_RATE,
        "Protect_KL":             PROTECT_KL,

        # --------------------
        # Auxiliary metrics
        # --------------------
        "Paraphrase_Robustness":  PARAPHRASE_ROBUSTNESS,
        "Paraphrase_Consistency": PARAPHRASE_CONSISTENCY,
        "Fluency_GE":             FLUENCY_GE,
    }
])


# ============================================================
# 28. FINAL PAPER-STYLE ROW
# ============================================================

PAPER_ROW = pd.DataFrame([
    {
        "Dataset":                "HumAID",
        "S":                      SCORE_S,
        "ES":                     ES,
        "RG":                     RG_AFTER,
        "RG_Gain":                RG_GAIN,
        "NS":                     NS,
        "Retention":              RETENTION_FINAL,
        "Forgetting":             FORGETTING_RATE,
        "Protect_KL":             PROTECT_KL,
        "Fluency_GE":             FLUENCY_GE,
        "Paraphrase_Robustness":  PARAPHRASE_ROBUSTNESS,
        "Paraphrase_Consistency": PARAPHRASE_CONSISTENCY,
    }
])


# ============================================================
# 29. SAVE EVERYTHING
# ============================================================

MAIN.to_csv(OUT_DIR / "00_MAIN_EDITING_METRICS.csv", index=False)
PAPER_ROW.to_csv(OUT_DIR / "02_FINAL_PAPER_ROW.csv", index=False)
scalability.to_csv(OUT_DIR / "03_SEQUENTIAL_SCALABILITY.csv", index=False)
pd.DataFrame([LOCALIZATION]).to_csv(OUT_DIR / "04_LOCALIZATION_SUMMARY.csv", index=False)


# ============================================================
# 30. PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 140)
print("FINAL EDITING METRICS")
print("=" * 140)
print(MAIN.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n")
print("=" * 140)
print("FINAL PAPER ROW")
print("=" * 140)
print(PAPER_ROW.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n")
print("=" * 140)
print("VALIDATION")
print("=" * 140)

print("Score S uses:")
print(f"ES={ES:.6f}, RG={RG_AFTER:.6f}, NS={NS:.6f}")
print(f"S={SCORE_S:.6f}")
print()
print("Unassigned samples:", len(UNASSIGNED_IDS))

if UNASSIGNED_IDS:
    print(sorted(UNASSIGNED_IDS))

print("\nSaved to:")
print(OUT_DIR)


CURRENT RUN: /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run
Final predictions : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HumAID_FINAL_W0_WT_generated.csv
Ripple sets : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HUMAID_ripple_sets.csv
Protect sets : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HUMAID_protect_sets.csv
Sequential instances : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HUMAID_seq_edit_generated_per_instance.csv
Sequential diagnostics : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/HUMAID_seq_edit_generated_diagnostic.csv
Paraphrase : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/FINAL_COMPLETE_METRICS/GENERALIZATION_PARAPHRASE_DETAILS.csv
Fluency : /scratch/project_2019963/ablation/Wo_KL_HUMAID/HUMAID_500_run/FINAL_COMPLETE_METRICS/FLUENCY_GE_500.csv
Evaluation samples: 500

Classes: 10
caution_and_advice -> caution_and_advice
displaced_people_and_evac